# Quick Start — reproducing the paper's figures

Welcome. This notebook is the **entry point** to the figure-reproduction suite for
*Complexity Analysis and Efficient Implementation of the Rodeo Algorithm for
Steady-State Preparation*. It does three things:

1. introduces the **model** and the two **filters** being compared,
2. shows the **core API** (`rodeo_ness`) used throughout,
3. gives a **map** of which notebook reproduces which paper figure, and *what model /
   what comparison* each one represents.

Every figure has its own notebook (`Figure01_*.ipynb` … `Figure13_*.ipynb`) with a
detailed write-up. Start here, then open whichever figure you want.


## 1. The physical setup in one minute

We prepare the **non-equilibrium steady state (NESS)** of an open quantum system — the
density matrix `ρ_ss` with `L(ρ_ss) = 0`, where `L` is the vectorized Liouvillian. The
trick (following Ramusat & Savona) is to embed the non-Hermitian `L` into a Hermitian
operator

```
M = [[ 0 , L  ],
     [ L†, 0  ]]
```

whose **zero sector** encodes the steady state. Isolating that zero sector is a spectral
filtering problem, and the paper compares two filters for it:

- **Phase estimation (QPE)** — resolve eigenvalues with a phase register; leakage falls
  as a power law in depth.
- **Rodeo** — repeated measurement-conditioned controlled evolutions centred at the known
  zero eigenvalue; residual weight falls exponentially in the number of cycles, i.e.
  **logarithmically** in the target error.

The single quantity that governs the comparison is the **spectral separation**
`g = min_{j≠0,1}|φ_j|` (the smallest nonzero singular value of `L`).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))          # repo root (rodeo_ness package)
sys.path.insert(0, os.path.abspath("../../reproduce/core"))  # original figure engine
import numpy as np
import rodeo_ness as rn
print("rodeo_ness", rn.__version__)


rodeo_ness 0.1.0


## 2. Core API tour

The `rodeo_ness` package builds the model and evaluates the filters. The single-spin
benchmark (`H = h σ_x`, jump `σ⁻`) has gap `g = 1/2` for all `h` and exact NESS
observables `⟨σ_y⟩ = 2/3`, `⟨σ_z⟩ = -1/3`.


In [2]:
# build the single-spin Liouvillian and inspect it
L = rn.single_spin_liouvillian(h=0.5)
print("spectral separation g =", rn.spectral_separation(L))   # 0.5
print("decay rate g_decay    =", rn.decay_rate(L))            # 0.5 (coincide here)

rho = rn.steady_state(L)
SY = np.array([[0,-1j],[1j,0]]); SZ = np.array([[1,0],[0,-1]])
print("exact <sy> =", round(float(np.real(np.trace(SY@rho))),4))
print("exact <sz> =", round(float(np.real(np.trace(SZ@rho))),4))


spectral separation g = 0.49999999999999944
decay rate g_decay    = 0.5
exact <sy> = -0.6667
exact <sz> = -0.3333


In [3]:
# the gate-cost ratio at a fixed depth (>1 means Rodeo is cheaper)
print("G_QPE / G_Rodeo  =", round(rn.cost_ratio(L, total_depth=15), 2))

# growing the chain CLOSES g, so the Rodeo advantage shrinks -- the key insight
for N in [1,2,3,4]:
    g = rn.spectral_separation(rn.tfim_liouvillian(N, J=0.0, gamma=1.0))
    print(f"  N={N}:  g = {g:.3f}")


G_QPE / G_Rodeo  = 1.1
  N=1:  g = 0.500
  N=2:  g = 0.432
  N=3:  g = 0.369
  N=4:  g = 0.312


## 3. Figure map — what each notebook reproduces

The figures fall into three groups by **what model** they use and **what they compare**.

### Group A — single-spin benchmark (exact, the R&S model)
*Engine: `reproduce/core/` (the original figure code).*

| Figure | Notebook | Model | Compares |
|---|---|---|---|
| **1** | `Figure01_cost` | single spin, `g=1/2` | QPE vs Rodeo (Gauss. + det.): **depth vs target precision** — power law vs log |
| **2** | `Figure02_restart` | single spin | restart cost: QPE terminal-measurement (`1/P`) vs Rodeo early-abort overhead |
| **3** | `Figure03_obs_error` | single spin, `h=0.5` | observable error of `⟨σ_y⟩` vs depth, all three schedules |
| **5** | `Figure05_observables` | single spin, `h∈{0.5,1,1.5}` | NESS recovery (2×2: QPE/Rodeo × `σ_y`/`σ_z`) — accuracy check |

### Group B — separation dependence (dissipative TFIM sweep)
*Engine: `rodeo_ness` package.*

| Figure | Notebook | Model | Compares |
|---|---|---|---|
| **4** | `Figure04_cost_vs_g` | TFIM, swept `(γ,N,J)` | **cost ratio vs `g`** — the collapse; Rodeo advantage grows with `g` |
| **10** | `Figure10_collapse` | TFIM `(γ,N,J)` | cost vs `g_decay` (scatter) vs `g` (collapse) — which gap matters |
| **11** | `Figure11_interacting` | interacting TFIM (`J>0`) | collapse survives interactions |
| **12** | `Figure12_density` | synthetic spectra | spectral density is subleading (saturates after ~6 modes) |
| **13** | `Figure13_two_gaps` | TFIM + dimension counting | scope: `g`↔`g_decay` correlation; quantum advantage is in `N` |

### Group C — circuit-level validation (Qiskit / AerSimulator)
*Engine: `rodeo_ness` + Qiskit (optional extra).*

| Figure | Notebook | Model | Compares |
|---|---|---|---|
| **6** | `Figure06_dynamic_circuit` | single-spin embedding | the compiled dynamic Rodeo circuit (measurement-conditioned cycles) |
| **7** | `Figure07_trotter` | single-spin `M` | Trotter decomposition into 9 Pauli-term evolutions |
| **8** | `Figure08_aer_convergence` | single spin on AerSimulator | circuit convergence to exact `⟨σ_z⟩=-1/3` |
| **9** | `Figure09_aer_cycle_saving` | single spin | gate-level early-abort saving (~35%) |

---

**To reproduce any figure:** open its notebook and run all cells. Each writes its PDF to
the working directory and explains the physics in more depth than the paper caption.

**Dependencies:** Groups A and B need only `numpy`, `scipy`, `matplotlib`. Group C needs
Qiskit (`pip install "rodeo_ness[circuit]"`); the AerSimulator notebooks fall back to an
exact-filter reference if Qiskit is absent.
